In [ ]:
# Data Preprocessing
# - Generate pseudo-masked 3D NIfTI images
# - Collect original/pseudo paths into a CSV (with pre/post pairing)
# - Train/Test split (stratified)
# - 5-fold StratifiedKFold splits

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

# Set image directory for pseudo generation
data_dir = ".../STRANet/Image/"


def random_block_mask_all_3d(img_data, block_size=(64, 64, 8), num_blocks=30):
    """
    Apply random block masking on a 3D volume by zeroing blocks.

    Parameters
    ----------
    img_data : np.ndarray
        3D array with shape (depth, height, width).
    block_size : tuple of int
        Size of the block (depth, height, width).
    num_blocks : int
        Number of blocks to zero out.

    Returns
    -------
    np.ndarray
        Masked 3D image with the same shape as input.
    """
    depth, height, width = img_data.shape
    block_depth, block_height, block_width = block_size

    assert block_depth <= depth and block_height <= height and block_width <= width, \
        "Block size must be smaller than image size."

    mask = np.ones_like(img_data)

    for _ in range(num_blocks):
        z = np.random.randint(0, depth - block_depth + 1)
        y = np.random.randint(0, height - block_height + 1)
        x = np.random.randint(0, width - block_width + 1)
        mask[z:z + block_depth, y:y + block_height, x:x + block_width] = 0

    masked_img = img_data * mask
    return masked_img

In [ ]:
# === Step 1: Generate pseudo images ===
for subfolder in os.listdir(data_dir):
    subfolder_path = os.path.join(data_dir, subfolder)

    if os.path.isdir(subfolder_path):
        for filename in os.listdir(subfolder_path):
            if filename.endswith(".nii.gz"):
                file_path = os.path.join(subfolder_path, filename)

                img = nib.load(file_path)
                img_data = img.get_fdata()

                pseudo_img_data = random_block_mask_all_3d(
                    img_data, block_size=(64, 64, 8), num_blocks=30
                )

                pseudo_img = nib.Nifti1Image(pseudo_img_data, img.affine, img.header)
                pseudo_filename = f"pseudo_{filename}"
                pseudo_file_path = os.path.join(subfolder_path, pseudo_filename)
                nib.save(pseudo_img, pseudo_file_path)

                print(f"Generated and saved: {pseudo_file_path}")

In [ ]:
# === Step 2: Collect original & pseudo paths into a CSV (with pre/post pairing) ===

def collect_data_paths(base_dir):
    """
    Collect paths of original and already-generated pseudo images and save to a DataFrame.

    Returns
    -------
    pd.DataFrame
        A merged table where each row corresponds to one patient, including
        pre/post original paths and pre/post pseudo paths.
    """
    results = []

    # Use two-class labels here; modify if needed
    for label in ['progress', 'stable']:
        label_dir = os.path.join(base_dir, label)

        # Set binary labels for two-class classification
        binary_label = 1 if label in ['progress'] else 0

        # Traverse original images in each label folder
        for file_name in os.listdir(label_dir):
            if file_name.endswith('.nii.gz') and not file_name.startswith('pseudo_'):
                parts = file_name.split('_')
                patient_id = parts[0]
                treatment_time = 'pre' if 'pre' in file_name else 'post'

                # Original image path
                file_path = os.path.join(label_dir, file_name)

                # Corresponding pseudo image path
                pseudo_file_name = f"pseudo_{file_name}"
                pseudo_file_path = os.path.join(label_dir, pseudo_file_name)

                # Check if pseudo exists
                if not os.path.exists(pseudo_file_path):
                    print(f"No matching pseudo file found: {pseudo_file_name}")
                    continue

                results.append({
                    'Patient_ID': patient_id,
                    'Label': label,
                    'Binary_Label': binary_label,
                    'Time': treatment_time,
                    'Path': file_path,
                    'Pseudo_Path': pseudo_file_path
                })

    df = pd.DataFrame(results)

    # Reshape to one row per patient with pre & post columns
    df_pre = df[df['Time'] == 'pre'] \
        .rename(columns={'Path': 'Pre_path', 'Pseudo_Path': 'Pre_pseudo_path'}) \
        .drop('Time', axis=1)

    df_post = df[df['Time'] == 'post'] \
        .rename(columns={'Path': 'Post_path', 'Pseudo_Path': 'Post_pseudo_path'}) \
        .drop('Time', axis=1)

    final_df = pd.merge(
        df_pre, df_post,
        on=['Patient_ID', 'Label', 'Binary_Label'],
        how='outer'
    )

    final_df.to_csv(".../STRANet/Data/Image_Path.csv", index=False)
    return final_df


# Call the function (keep your original paths)
base_dir = ".../STRANet/Image/"
final_data = collect_data_paths(base_dir)
print(final_data)


In [ ]:
# === Step 3: Stratified Train/Test split ===
datflag = pd.read_csv(".../STRANet/Data/Image_Path.csv")

train_size = 0.8
test_size = 0.2
random_state = 42

train_df, test_df = train_test_split(
    datflag,
    train_size=train_size,
    test_size=test_size,
    random_state=random_state,
    stratify=datflag['Label']
)

output_dir = ".../STRANet/Data/"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(f"{output_dir}/train_data.csv", index=False)
test_df.to_csv(f"{output_dir}/test_data.csv", index=False)

print("Training Data:")
print(train_df.head())

print("Testing Data:")
print(test_df.head())


In [ ]:
# === Step 4: 5-fold StratifiedKFold from training set ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
fold_num = 1

def save_train_val_split(fold_dir, train_split, val_split):
    train_split.to_csv(f"{fold_dir}/train_split.csv", index=False)
    val_split.to_csv(f"{fold_dir}/val_split.csv", index=False)

labels = train_df['Label']

for train_index, val_index in skf.split(train_df, labels):
    print(f"Saving train and validation data for fold {fold_num}...")

    fold_dir = f"{output_dir}/fold_{fold_num}/"
    os.makedirs(fold_dir, exist_ok=True)

    train_split = train_df.iloc[train_index]
    val_split = train_df.iloc[val_index]

    save_train_val_split(fold_dir, train_split, val_split)

    fold_num += 1

print("Five-fold cross-validation splits saved.")